# **Miniproject 2**
## **~Large~ Small Language Model**

### **Objective**
Implement a transformer-based, character-level language model (GPT-like) and train it on the Shakespeare dataset. By the end of this project, you should be able to generate Shakespearean-like text given a seed string.

You will probably want to train the model on a GPU. You can use free GPUs on [Google Colab](https://colab.research.google.com/?utm_source=scs-index).

### **Dataset**:

The Shakespeare dataset contains the complete works of William Shakespeare, including his plays, poems, and sonnets.

[**Download link**](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt)

In a character-level language model, each character in the input data is mapped to its respective index from a dictionary. The input to the model is in the form (B, N), where B is the batch size and N is the number of tokens for each sequence. The model was tested with B=N=128, but feel free to explore different values.

An interface for the dataset class that takes care of tokenization is provided below.



```python
from torch.utils.data import Dataset

class CharDataset(Dataset):
    """
    Emits batches of characters.

    Adapted from "https://github.com/karpathy/minGPT".
    """

    def __init__(self, config, data):

        chars = ... # get characters from the input data
        self.stoi = { ch:i for i,ch in enumerate(chars) } # map characters to integer indices

        ...

    def get_vocab_size(self):
        raise NotImplementedError()

    def __len__(self):
        raise NotImplementedError()

    def __getitem__(self, idx):
        # grab a chunk of (block_size + 1) characters from the data
        # encode every character to an integer
        # return the chunk and the shifted version as tensors
        pass
```




### **Requirements**

#### **Architecture**

Implement the Transformer's decoder-only structure.
This includes

* input token embeddings
* the causal multi-head self-attention mechanism
* feed-forward neural networks
* positional encodings, residual connections, layer normalizations.

The project was tested with $12$ layers, $8$ attention heads, and $768$ embedding dimensions, on a single GPU.

The `forward` method for the entire model has the following form:

```
tok_emb = WTE(idx) # token embeddings
pos_emb = WPE(pos) # position embeddings
x = Dropout(tok_emb + pos_emb)
for Block in Blocks:
    x = Block(x)
x = Final_LayerNorm(x)
logits = LM_Head(x)
```

The `forward` method for the transformer block has the following form:



```
x = x + self.CausalSelfAttn(self.LayerNorm_1(x))
out = x + self.MLP(self.LayerNorm_2(x))
```

---

#### **Training**

In a character-level transformer language model, the goal is to predict the next character in a sequence given the previous characters. To train such a model effectively, we use two versions of our data: the input sequence and a shifted version of this sequence, which serves as the target for our predictions.

Preprocess the dataset to a character-level representation.
Use a sliding window approach for sequence chunks (e.g., window size of $128$ characters).
Implement causal masking for the self-attention mechanism.
Use the [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) optimizer and the cross-entropy loss.

**Optional**:

* Implement a learning rate decay strategy
* Implement gradient clipping

---


#### **Evaluation and Inference**

* Monitor the cross-entropy loss. Use a seed string to initialize the model and generate Shakespearean-like text.

* In order to generate the characters, at each generation step you can either select the character with the highest probability, or you can sample according to the output distribution.

The high-level pseudocode for generation is:

```python
model.eval()
with torch.no_grad():
    context = "O God, O God!"
    tokenized_context = tokenize(context)
    # the model should implement a method to generate tokens given a prompt
    y = model.generate(tokenized, ...)
    completion = tokens_to_string(y)
```

**Optional**:
* Compute the [perplexity](https://medium.com/@priyankads/perplexity-of-language-models-41160427ed72#:~:text=Intuitively%2C%20perplexity%20means%20to%20be,loss%20obtained%20from%20the%20model.) metric for quantitative evaluation.

### **Example Outputs**

The following are my outputs after $6000$ steps of training, with the seed string "O God, O God!"



```
O God, O God! neither? unto the base very ears,
As damned with it.

DUKE OF YORK:
Away! Once more, one word.

RICHARD:
Clove, dear so; and therein my son will be
false of woe: if ye seems to be the mother
Of gracious order this time when R going kinsperse eyes,
What dost bewreck her fairer drying tears.

NORTHUMBERLAND:
Have you forgot the Duke of Norfolk, get him to
again; and and agilic: there is my spirit
So maly did must such a marble perfection.

ELBOW:
Come, bring them with oaths, and so deliver
```


### Resources:

* Vaswani et al., "Attention is All You Need": [link](https://arxiv.org/abs/1706.03762)

* Illustrated Transformer by Jay Alammar: [link](https://jalammar.github.io/illustrated-transformer/)

* OpenAI GPT-2 Paper: [link](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

* Deep Learning Course slides on transformers: [link](https://fleuret.org/dlc/materials/dlc-handout-13-3-transformers.pdf)

In [1]:
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from architecture import Shakespeare

In [2]:
class CharDataset(Dataset):

    def __init__(self, config, data):
        self.block_size = config.block_size

        with open(data, "r", encoding="utf-8") as f:
            self.data = f.read()

        chars = sorted(list(set(self.data)))
        self.char_to_num = {ch: i for i, ch in enumerate(chars)}
        self.num_to_char = {i: ch for ch, i in self.char_to_num.items()}
        self.vocab_size = len(chars)

        self.data_encoded = torch.tensor([self.char_to_num[c] for c in self.data], dtype=torch.long)

    def get_vocab_size(self):
        return self.vocab_size

    def __len__(self):
        return len(self.data_encoded) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data_encoded[idx : idx + self.block_size + 1]

        x = chunk[:-1]
        y = chunk[1:]

        return x, y


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

block_size = 128

dataset = CharDataset(type("cfg", (), {"block_size": block_size})(), "Shakespeare.txt")

loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = Shakespeare(
    vocab_size=dataset.get_vocab_size(),
    block_size=block_size,
    embed_dim=768,
    num_heads=8,
    n_layers=12
).to(device)

base_lr = 0.0003
optimizer = torch.optim.Adam(model.parameters(), lr=base_lr)
total_step = 60
for i, (x, y) in enumerate(loader) :
    x = x.to(device)
    y = y.to(device)

    optimizer.zero_grad()
    _, loss = model(x, y)
    loss.backward()
    clip_grad_norm_(model.parameters(), 1)
    optimizer.step()
    lr = base_lr * (1 - (i / total_step))
    for param_group in optimizer.param_groups :
        param_group['lr'] = lr

    if i % 10 == 0 :
        print(f"step {i}, loss = {loss.item():.4f}")

    if i == total_step :
        break


step 0, loss = 4.3070
step 10, loss = 3.2023
step 20, loss = 2.7743
step 30, loss = 2.5832
step 40, loss = 2.5185
step 50, loss = 2.5215
step 60, loss = 2.5125


In [5]:
model.eval()
with torch.no_grad():
    context = "O God, O God!"
    tokenized_context = torch.tensor([[dataset.char_to_num[c] for c in context]], device=device)
    # the model should implement a method to generate tokens given a prompt
    y = model.generate(tokenized_context, max_new_tokens=300)
    completion = "".join(dataset.num_to_char[i] for i in y[0].tolist())


In [6]:
print(completion)

O God, O God! the pallrndunwhee mer, sityo heracl gay my han, fe paned,

Thieverid, stesoTan tishendarant'd t nd mice m t, wh puuighous telutan-
Dolien thorve mfits thee h fof l then frenowhis athed plis;
Fo thy the'de deserang.
Wirthis. n her hed frdrolownrifo llo g athouss my y wy;
SAR: cer:
Whes IENAnther th 
